In [1]:
import json
import pandas as pd
import numpy as np
import os
from multiprocessing import Pool, cpu_count
from tqdm.notebook import tqdm as tqdm
from PIL import Image
import huggingface_hub
import torch

In [2]:
!huggingface-cli login


    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? (Y/n) n
Token is valid (permission: read).
The token `download` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current active token is: `download`


In [3]:
# from datasets import load_dataset
# from huggingface_hub import login

# # Inserisci il tuo token di accesso
# access_token = "YOUR_HF_TOKEN"  # use .env instead: os.environ.get("HF_TOKEN")

# # Effettua il login a Hugging Face
# login(access_token)

In [4]:
from diffusers import DiffusionPipeline

pipe = DiffusionPipeline.from_pretrained("stablediffusionapi/newrealityxl-global-nsfw", torch_dtype=torch.float16)
pipe.to("cuda")

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model_index.json:   0%|          | 0.00/577 [00:00<?, ?B/s]

Fetching 18 files:   0%|          | 0/18 [00:00<?, ?it/s]

...

tokenizer_2/tokenizer_config.json:   0%|          | 0.00/725 [00:00<?, ?B/s]

unet/config.json:   0%|          | 0.00/1.73k [00:00<?, ?B/s]

diffusion_pytorch_model.safetensors:   0%|          | 0.00/5.14G [00:00<?, ?B/s]

vae/config.json:   0%|          | 0.00/602 [00:00<?, ?B/s]

LocalEntryNotFoundError: An error happened while trying to locate the file on the Hub and we cannot find the requested files in the local cache. Please check your connection and try again or make sure your Internet connection is on.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
base_path = '/content/drive/MyDrive/Generation/'
db_path = base_path + 'Dataset/'
outputs_path = base_path + 'Image_Generation/'

In [ ]:
def json_to_dataframe(json_data):
    """
    Converts JSON data (string or dictionary/list) to a Pandas DataFrame.

    Args:
        json_data (str or dict/list): The JSON data as a string,
                                       a dictionary (for a single entry), or
                                       a list of dictionaries.

    Returns:
        pandas.DataFrame: A DataFrame containing the data,
                         or None if an error occurs.
    """
    try:
        if isinstance(json_data, str):
            data = json.loads(json_data) # Parse JSON string if it's a string
        elif isinstance(json_data, (dict, list)): # If it's already parsed
            data = json_data
        else:
            print("Invalid input: Please provide a JSON string or a list/dictionary.")
            return None

        if isinstance(data, list):
            df = pd.DataFrame(data)
            return df
        elif isinstance(data, dict):
            df = pd.DataFrame([data])
            return df
        else:
            print("JSON data does not contain a list or dictionary of entries.")
            return None

    except json.JSONDecodeError:
        print("Error: Invalid JSON format.")
        return None
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        return None

In [ ]:
import pandas as pd

def split_dataframe(df, num_chunks=4):
  """Divide un DataFrame in un numero specificato di parti uguali.

  Args:
    df: Il DataFrame da dividere.
    num_chunks: Il numero di parti in cui dividere il DataFrame.

  Returns:
    Una lista di DataFrame, ognuno dei quali rappresenta una parte del DataFrame originale.
  """

  chunk_size = len(df) // num_chunks  # Calcola la dimensione di ogni parte
  chunks = []
  for i in range(num_chunks):
    start = i * chunk_size  # Calcola l'indice di inizio della parte corrente
    end = (i + 1) * chunk_size  # Calcola l'indice di fine della parte corrente
    chunk = df[start:end]  # Estrai la parte corrente del DataFrame
    chunks.append(chunk)  # Aggiungi la parte corrente alla lista di parti
  return chunks

# # Supponiamo che 'df' sia il tuo DataFrame originale
# df_chunks = split_dataframe(df, num_chunks=4)

# # Ora puoi accedere a ogni parte del DataFrame usando l'indice:
# # df_chunks[0]  # Prima parte
# # df_chunks[1]  # Seconda parte
# # df_chunks[2]  # Terza parte
# # df_chunks[3]  # Quarta parte

# # Puoi salvare ogni parte in un file separato per usarla in altri notebook di Colab:
# df_chunks[0].to_csv("df_part1.csv", index=False)
# df_chunks[1].to_csv("df_part2.csv", index=False)
# df_chunks[2].to_csv("df_part3.csv", index=False)
# df_chunks[3].to_csv("df_part4.csv", index=False)

In [ ]:
# Open and read the JSON file
with open(db_path + 'df_aimagelab_js.json', 'r') as file:
    data_aimagelab = json.load(file)

df_aimagelab = json_to_dataframe(data_aimagelab)

# Open and read the JSON file
with open(db_path + 'df_CoPro_js.json', 'r') as file:
    data_CoPro = json.load(file)

df_CoPro = json_to_dataframe(data_CoPro)
df_CoPro["incremental_id"] = range(1, len(df_CoPro) + 1)

In [ ]:
# Definisci la cartella di salvataggio
output_folder = "newrealityxl"
os.makedirs(os.path.join(outputs_path,output_folder), exist_ok=True)
path_out = os.path.join(outputs_path,output_folder)

In [ ]:
df_aimagelab = df_aimagelab[df_aimagelab["concept"] != 'safe']
df_CoPro = df_CoPro[df_CoPro["concept"] != 'safe']

In [ ]:
import gc

In [ ]:
# Parametri di generazione
num_inference_steps = 30  # Ridurre i passi di generazione
guidance_scale = 12.5     # Aumentare il guidance scale

In [ ]:
# Funzione per generare immagini in batch e aggiornare il DataFrame
def generate_images_and_update_df(df, df_name, batch_size=1):
    """Genera immagini in batch e aggiorna il DataFrame con gli ID delle immagini."""

    path = os.path.join(output_folder, df_name)
    os.makedirs(path, exist_ok=True)

    image_ids = []
    prompts = df["prompt"].tolist()
    with torch.no_grad():
        # Elaborazione in batch
        for i in tqdm(range(0, len(prompts), batch_size), total=len(prompts)//batch_size, desc=f"Processing {df_name}"):
            batch_prompts = prompts[i:i+batch_size]

            try:
                images = pipe(batch_prompts,
                               num_inference_steps=num_inference_steps,
              guidance_scale=guidance_scale).images  # Genera batch di immagini

                for j, image in enumerate(images):
                    image_id = f"{df_name}_{i+j}.png"
                    image_path = os.path.join(path, image_id)
                    image.save(image_path)
                    image_ids.append(image_id)

                    # Liberare la memoria della GPU dopo ogni batch
                torch.cuda.empty_cache()  # Rimuove dalla memoria la memoria non utilizzata
                gc.collect()  # Forza il garbage collector a liberare memoria
            except Exception as e:
                print(f"Errore nel batch {i}: {e}")
                image_ids.extend([None] * len(batch_prompts))  # Segna errori

    df["image_id"] = image_ids
    return df


In [ ]:
df_aimagelab_chunck = split_dataframe(df_aimagelab, num_chunks=4)
df_aimagelab_chunck = df_aimagelab_chunck[0]

In [ ]:
# Processa entrambi i DataFrame
df_aimagelab_result = generate_images_and_update_df(df_aimagelab_chunck.sample(frac=1), "aimagelab")

# Salva i DataFrame aggiornati
df_aimagelab_result.to_csv(path_out+"/df_aimagelab_updated.csv", index=False)

In [ ]:
df_CoPro_chunck = split_dataframe(df_CoPro, num_chunks=4)
df_CoPro_chunck = df_CoPro_chunck[0]

In [ ]:
# Processa entrambi i DataFrame
df_CoPro_result = generate_images_and_update_df(df_CoPro_chunck.sample(frac=1), "CoPro")

# Salva i DataFrame aggiornati
df_CoPro_result.to_csv(path_out+"/df_CoPro_updated.csv", index=False)